In [173]:
import gc
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime

# Filtragem inicial dos arquivos originais

A célula seguinte filtra os arquivos .csv do ano 2019 até 2026 e utiliza somente os casos da Paraíba (código 25, padronizado pelo IBGE).

A razão da não utilização dos dados dos anos anteriores se dão por mudanças nos critérios de diagnósticos feitos pelo Sistema de Informação de Agravos de Notificação (SINAN), recirculação de sorotipos do vírus (DENV-1, DENV-2, DENV-3, DENV-4), mudanças na metodologia de coleta dos dados, entre outros fatores que poderiam gerar inconsistências entre períodos mais longos. Além disso, evita-se utilizar volumes excessivos de dados.

In [174]:
"""def ler_filtrado(arquivo):
    chunks = []
    for chunk in pd.read_csv(arquivo, chunksize=50000):
        chunks.append(chunk[chunk['SG_UF'] == 25])
        gc.collect()
    return pd.concat(chunks, ignore_index=True)

dfs = []

for ano in range(19, 27):
    arquivo = f'data/DENGBR{ano:02d}.csv'
    df_temp = ler_filtrado(arquivo)
    dfs.append(df_temp)
    del df_temp
    gc.collect()
    print(f"{arquivo} OK — {len(dfs[-1])} linhas")

df = pd.concat(dfs, ignore_index = True)"""

'def ler_filtrado(arquivo):\n    chunks = []\n    for chunk in pd.read_csv(arquivo, chunksize=50000):\n        chunks.append(chunk[chunk[\'SG_UF\'] == 25])\n        gc.collect()\n    return pd.concat(chunks, ignore_index=True)\n\ndfs = []\n\nfor ano in range(19, 27):\n    arquivo = f\'data/DENGBR{ano:02d}.csv\'\n    df_temp = ler_filtrado(arquivo)\n    dfs.append(df_temp)\n    del df_temp\n    gc.collect()\n    print(f"{arquivo} OK — {len(dfs[-1])} linhas")\n\ndf = pd.concat(dfs, ignore_index = True)'

Ao final da filtragem, criamos o arquivo .csv do dataset com a primeira limpeza.

In [175]:
# df.to_csv('data/dengue_paraiba_2019-2026.csv', index = False)

In [176]:
df = pd.read_csv('data/dengue_paraiba_2019-2026.csv')

/tmp/ipykernel_102382/1392270464.py:1: DtypeWarning: Columns (11,22,44,45,46,52,101,120) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/dengue_paraiba_2019-2026.csv')


In [177]:
df.info()
df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111145 entries, 0 to 111144
Columns: 122 entries, TP_NOT to MIGRADO_W
dtypes: float64(96), int64(6), object(20)
memory usage: 103.5+ MB


,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,DT_SIN_PRI,...,PLAQ_MENOR,CON_FHD,COMPLICA,TP_SISTEMA,NDUPLIC_N,CS_FLXRET,FLXRECEBI,ANO_NASC,DT_DIGITA,MIGRADO_W
0,2,A90,2019-05-15,201920,2019,25,250290,1424.0,7256930.0,2019-05-13,...,NaN,NaN,NaN,2.0,NaN,1.0,NaN,NaN,NaN,NaN
1,2,A90,2019-05-16,201920,2019,25,250290,1424.0,7256930.0,2019-05-12,...,NaN,NaN,NaN,2.0,NaN,1.0,NaN,NaN,NaN,NaN
2,2,A90,2019-01-31,201905,2019,25,251080,1422.0,5487684.0,2019-01-24,...,NaN,NaN,NaN,2.0,NaN,1.0,NaN,NaN,NaN,NaN
3,2,A90,2019-05-20,201921,2019,25,251080,1422.0,5487684.0,2019-05-12,...,NaN,NaN,NaN,2.0,NaN,0.0,NaN,NaN,NaN,NaN
4,2,A90,2019-08-28,201935,2019,25,251080,1422.0,2605317.0,2019-08-27,...,NaN,NaN,NaN,2.0,NaN,0.0,NaN,NaN,NaN,NaN
5,2,A90,2019-07-10,201928,2019,25,251080,1422.0,2605481.0,2019-07-06,...,NaN,NaN,NaN,2.0,NaN,0.0,NaN,NaN,NaN,NaN
6,2,A90,2019-07-22,201930,2019,25,251080,1422.0,2605481.0,2019-07-19,...,NaN,NaN,NaN,2.0,NaN,1.0,NaN,NaN,NaN,NaN
7,2,A90,2019-05-02,201918,2019,25,251080,1422.0,2605481.0,2019-04-30,...,NaN,NaN,NaN,2.0,NaN,0.0,NaN,NaN,NaN,NaN
8,2,A90,2019-05-09,201919,2019,25,251080,1422.0,2605481.0,2019-05-02,...,NaN,NaN,NaN,2.0,NaN,1.0,NaN,NaN,NaN,NaN
9,2,A90,2019-09-27,201939,2019,25,251080,1422.0,2605481.0,2019-09-21,...,NaN,NaN,NaN,2.0,NaN,0.0,NaN,NaN,NaN,NaN


In [178]:
# Altera os nomes das colunas para letras minúsculas para facilitar na utilização do código
df.columns = df.columns.str.lower()

# Remoção de colunas de pouco valor preditivo

Algumas colunas possuem nenhum ou pouco valor preditivo, por exemplo, não queremos que a coluna sg_uf_not (UF que o caso foi notificado) tenha alguma influência no modelo, tendo em vista que o dataset utiliza apenas dados da Paraíba.

Outras colunas possuem informações administrativas do próprio sistema do SINAN.

Outras são relacionadas à Chikungunya.

Todas as informações sobre as colunas estão descritas no dicionário de dados disponibilizado.

In [179]:
# Remoção de colunas sem valor preditivo
df.drop(columns=['tp_not', 'id_agravo', 'dt_notific', 'sem_not',
                  'id_regiona', 'id_unidade', 'dt_invest', 'id_ocupa_n',
                  'dt_digita', 'cs_flxret', 'flxrecebi', 'nduplic_n', 'migrado_w', 'tp_sistema'], axis=1, inplace=True, errors='ignore')

# Remoção de colunas relacionadas à Chikungunya
df.drop(columns=['dt_chik_s1', 'dt_chik_s2', 'dt_prnt', 'res_chiks1',
                 'res_chiks2', 'resul_prnt', 'clinc_chik'], axis=1, inplace=True, errors='ignore')

# Remoção de colunas por data leakage
df.drop(columns=['alrm_hipot', 'alrm_plaq', 'alrm_vom', 'alrm_sang', 'alrm_hemat', 
                 'alrm_abdom', 'alrm_letar', 'alrm_hepat', 'alrm_liq', 'dt_alrm', 
                 'grav_pulso', 'grav_conv', 'grav_ench', 'grav_insuf', 'grav_taqui', 
                 'grav_extre', 'grav_hipot', 'grav_hemat', 'grav_melen', 'grav_metro', 
                 'grav_sang', 'grav_ast', 'grav_mioc', 'grav_consc', 'grav_orgao', 
                 'dt_grav', 'evolucao', 'dt_obito', 'hospitaliz', 'dt_interna', 
                 'con_fhd', 'complica', 'dt_encerra'], axis=1, inplace=True, errors='ignore')

# Remoção de colunas de localização
df.drop(columns=['sg_uf_not', 'id_municip', 'sg_uf', 'id_rg_resi', 
                 'id_pais', 'coufinf', 'copaisinf', 'comuninf', 'municipio', 
                 'tpautocto'], axis=1, inplace=True, errors='ignore')

# Remoções de datas
df.drop(columns=['sem_pri', 'ano_nasc', 'dt_soro', 'dt_ns1',
                'dt_viral', 'dt_pcr'], axis=1, inplace=True, errors='ignore')


# Tratando a coluna idade

A composição dos valores idade seguem o seguinte critério:

1º dígito:

1. Hora
2. Dia
3. Mês
4. Ano

Ex: 3009 – nove meses, 4018 – dezoito anos, 2019 – dezenove dias

Primeiro é necessário corrigir idades que não batem com a data de nascimento e o ano da notificação

In [180]:
def corrigir_idade(row):
    try:
        # Obtém o ano da notificação
        ano_notificacao = int(row['nu_ano'])

        dt_nasc = pd.to_datetime(row['dt_nasc'], format='%Y-%m-%d', errors='coerce')

        if pd.isna(dt_nasc):
            return row['nu_idade_n'] # Mantém o valor original se não houver data
        
        dt_referencia = pd.Timestamp(year=ano_notificacao, month=1, day=1)

        idade_calculada = (dt_referencia - dt_nasc).days / 365.25

        if idade_calculada >= 1:
            nu_idade_corrigido = 4000 + int(idade_calculada)
        elif idade_calculada >= 1/12:  # mais de 1 mês
            nu_idade_corrigido = 3000 + int(idade_calculada * 12)
        elif idade_calculada >= 1/365:  # mais de 1 dia
            nu_idade_corrigido = 2000 + int(idade_calculada * 365)
        else:  # horas
            nu_idade_corrigido = 1000 + int(idade_calculada * 8760)
        
        return nu_idade_corrigido
    
    except Exception:
        return row['nu_idade_n']  # mantém original em caso de erro
        


In [181]:
df['nu_idade_n'] = df.apply(corrigir_idade, axis=1)

Depois convertemos as idades para anos

In [182]:
def converter_idade_anos(valor):
    if pd.isna(valor):
        return None
    
    valor = int(valor)
    
    if valor <= 1000:
        return valor
    
    unidade = valor // 1000  # primeiro dígito
    quantidade = valor % 1000  # três últimos dígitos
    
    if unidade == 4:  # anos
        return quantidade
    elif unidade == 3:  # meses
        return round(quantidade / 12, 2)
    elif unidade == 2:  # dias
        return round(quantidade / 365, 2)
    elif unidade == 1:  # horas
        return round(quantidade / 8760, 2)
    else:
        return None

In [183]:
df['idade_anos'] = df['nu_idade_n'].apply(converter_idade_anos)

# Tratando linhas duplicadas

Para remoção de duplicatas, a ideia inicial era utilizar o campo `nduplic_n` do próprio SINAN, que classifica registros marcados pelo sistema como duplicidade, porém, observou-se que o campo `nduplic_n` estava vazio em todas as ocorrências. Foi decidido então utilizar colunas chaves para identificá-las, considerando como duplicata registros com mesma data de primeiros sintomas, idade, sexo, raça e município de residência.

In [184]:
colunas_chave = [
    'dt_sin_pri',  # data dos primeiros sintomas
    'nu_idade_n',  # idade
    'cs_sexo',     # sexo
    'id_mn_resi',  # município de residência
    'cs_raca'      # raça
]

In [185]:
duplicatas = df[df.duplicated(subset=colunas_chave, keep=False)]
print(f"Total de linhas duplicadas: {len(duplicatas)}")
print(duplicatas[colunas_chave].sort_values(by=colunas_chave).head(20))

# Verificar quantas linhas têm NaN em pelo menos uma coluna chave
nan_em_chave = df[colunas_chave].isna().any(axis=1)
print(f"Linhas com NaN em alguma coluna chave: {nan_em_chave.sum()}")

# Cruzar: dessas, quantas aparecem como duplicatas
print(f"Duplicatas que possuem NaN em coluna chave: {duplicatas[colunas_chave].isna().any(axis=1).sum()}")

Total de linhas duplicadas: 9609
       dt_sin_pri  nu_idade_n cs_sexo  id_mn_resi  cs_raca
5344   2019-01-02      4027.0       M    250750.0      4.0
5982   2019-01-02      4027.0       M    250750.0      4.0
13037  2019-01-03      4034.0       M    250750.0      4.0
13040  2019-01-03      4034.0       M    250750.0      4.0
2498   2019-01-11      4020.0       M    250750.0      4.0
9479   2019-01-11      4020.0       M    250750.0      4.0
7076   2019-01-17      4024.0       M    250750.0      4.0
7079   2019-01-17      4024.0       M    250750.0      4.0
8971   2019-01-25      4023.0       M    250750.0      9.0
8990   2019-01-25      4023.0       M    250750.0      9.0
5394   2019-01-25      4025.0       M    250750.0      9.0
28662  2019-01-25      4025.0       M    250750.0      9.0
15091  2019-01-28      4044.0       M    251530.0      4.0
18685  2019-01-28      4044.0       M    251530.0      4.0
2866   2019-01-30      4027.0       F    250750.0      4.0
10187  2019-01-30      

Observa-se a ocorrência de 9609 linhas duplicadas utilizando os critérios explicados acima, além disso não foi observada nenhuma duplicata com NaN em colunas chave. Será aplicado o `drop_duplicates()` para removê-las.

In [186]:
df = df.drop_duplicates(subset=colunas_chave, keep='first')

In [187]:
df[df.duplicated()]

,nu_ano,dt_sin_pri,dt_nasc,nu_idade_n,cs_sexo,cs_gestant,cs_raca,cs_escol_n,id_mn_resi,febre,...,gengivo,metro,petequias,hematura,sangram,laco_n,plasmatico,evidencia,plaq_menor,idade_anos


Agora podemos remover as colunas que foram utilizadas como chave na remoção de duplicatas.

In [188]:
df.drop(columns=['dt_sin_pri', 'id_mn_resi'], axis=1, inplace=True, errors='ignore')

Foram identificados três observações com valores da coluna nu_idade_n sendo 0, não sendo possível corrigir, foi tomada a decisão de removê-los. Tendo em vista que não impactaria significativamente no dataset

In [189]:
df[df['nu_idade_n'] == 0]

,nu_ano,dt_nasc,nu_idade_n,cs_sexo,cs_gestant,cs_raca,cs_escol_n,febre,mialgia,cefaleia,...,gengivo,metro,petequias,hematura,sangram,laco_n,plasmatico,evidencia,plaq_menor,idade_anos
74602,2022,NaN,0.0,M,6.0,4.0,10.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
75933,2022,NaN,0.0,M,6.0,1.0,5.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
79056,2022,NaN,0.0,F,NaN,4.0,9.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [190]:
df = df.drop(df[df['nu_idade_n'] == 0].index)

Agora que convertemos a idade para anos, vamos remover a coluna nu_idade_n do dataframe

In [191]:
df.drop(columns=['nu_idade_n'], axis=1, inplace=True, errors='ignore')

In [192]:
df.head(10)

,nu_ano,dt_nasc,cs_sexo,cs_gestant,cs_raca,cs_escol_n,febre,mialgia,cefaleia,exantema,...,gengivo,metro,petequias,hematura,sangram,laco_n,plasmatico,evidencia,plaq_menor,idade_anos
0,2019,1971-12-30,F,5.0,1.0,6.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.0
1,2019,2008-06-26,F,6.0,1.0,4.0,1.0,2.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0
2,2019,1995-12-18,F,2.0,1.0,6.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0
3,2019,1988-05-18,F,6.0,1.0,8.0,2.0,2.0,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0
4,2019,1979-08-30,M,6.0,4.0,NaN,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.0
5,2019,2013-11-03,M,6.0,1.0,10.0,1.0,1.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
6,2019,2009-03-23,M,6.0,1.0,6.0,1.0,1.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.0
7,2019,2014-07-02,M,6.0,4.0,10.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
8,2019,2007-02-21,F,5.0,1.0,5.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0
9,2019,2013-11-13,F,6.0,1.0,10.0,1.0,1.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0


In [193]:
df[df.duplicated()]

,nu_ano,dt_nasc,cs_sexo,cs_gestant,cs_raca,cs_escol_n,febre,mialgia,cefaleia,exantema,...,gengivo,metro,petequias,hematura,sangram,laco_n,plasmatico,evidencia,plaq_menor,idade_anos
821,2019,1976-02-06,F,9.0,9.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.0
1622,2019,2002-12-03,F,9.0,9.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0
2105,2019,1996-04-18,M,6.0,4.0,9.0,1.0,1.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0
2132,2019,1984-06-22,F,5.0,9.0,9.0,1.0,1.0,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.0
2396,2019,1991-08-26,F,9.0,4.0,9.0,1.0,1.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111037,2026,NaN,F,9.0,4.0,9.0,1.0,1.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.0
111045,2026,NaN,F,5.0,1.0,NaN,1.0,1.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.0
111063,2026,NaN,M,6.0,4.0,6.0,1.0,1.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.0
111091,2026,NaN,M,6.0,4.0,9.0,1.0,1.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.0
